# NB 04 — Scalable Feature Engineering

**Phase 3 — Tasks 3.1 / 3.2 / 3.3**

## Assigned features — formula  $f((k×n)+i)$,  n=21

| Student | k=0 | k=1 | k=2 | k=3 | k=4 | k=5 |
|---------|-----|-----|-----|-----|-----|-----|
| i=1     | f1  | f22 | f43 | f64 | f85 | f106 |
| i=4     | f4  | f25 | f46 | f67 | f88 | f109 |
| i=13    | f13 | f34 | f55 | f76 | f97 | —    |


In [ ]:
# ── Session bootstrap ─────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, importlib
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection')
SRC_DIR = PROJECT_ROOT / 'src'

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# src/__init__.py is committed to the repo, no runtime touch needed
print('PROJECT_ROOT:', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection


In [ ]:
!pip install -q camel-tools >=1.5.2

In [ ]:
import shutil, importlib
import src.feature_engineering as fe_mod
importlib.reload(fe_mod)

from src.utils import (
    create_spark_session, load_checkpoint, save_checkpoint,
    checkpoint_exists, add_src_to_spark, Timer,
    FIGURES_DIR, MODELS_DIR, DATA_PROCESSED, FEAT_PREFIX, CAMEL_CACHE,
)
from src.feature_engineering import (
    extract_all_features, build_tfidf_pipeline, build_word2vec_pipeline,
    assemble_feature_vector, FEATURE_COLS, print_mapreduce_design,
    _FEATURE_REGISTRY,
)
from src.data_preparation import stratified_split
from pyspark.sql import functions as F
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

os.environ["CAMELTOOLS_DATA"] = str(CAMEL_CACHE)
print("CAMELTOOLS_DATA:", os.environ["CAMELTOOLS_DATA"])
print("Cache exists:", CAMEL_CACHE.exists())

spark = create_spark_session('ArabicAIDetection_Features')
add_src_to_spark(spark)


CAMELTOOLS_DATA: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/.camel_cache
Cache exists: True


'/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/src_package.zip'

## 1. Optionally clean old feature checkpoints

In [ ]:
FORCE_REBUILD_FEATURES = False   # ← set True after changing feature_engineering.py

FEATURE_CHECKPOINTS = [
    'features_stylometric', 'features_tfidf', 'features_w2v',
    'split_train', 'split_val', 'split_test',
    'split_train_assembled', 'split_val_assembled', 'split_test_assembled',
]
MODEL_CHECKPOINTS = ['tfidf_pipeline', 'w2v_pipeline']

if FORCE_REBUILD_FEATURES:
    for name in FEATURE_CHECKPOINTS:
        p = DATA_PROCESSED / name
        if p.exists():
            shutil.rmtree(p)
            print('Deleted:', p)
    for name in MODEL_CHECKPOINTS:
        p = MODELS_DIR / name
        if p.exists():
            shutil.rmtree(p)
            print('Deleted model:', p)


## 2. Show feature registry

In [ ]:
print('Feature columns to be computed:')
for key, (_, rtype, col_name) in _FEATURE_REGISTRY.items():
    print(f'  {col_name:50s}  key={key:28s}  type={rtype}')
print(f'\nTotal: {len(FEATURE_COLS)} stylometric features')


Feature columns to be computed:
  feat_f1_total_chars                                 key=f1_total_chars                type=IntegerType()
  feat_f22_word_entropy                               key=f22_word_entropy              type=DoubleType()
  feat_f43_num_nouns                                  key=f43_num_nouns                 type=IntegerType()
  feat_f64_num_nominatives                            key=f64_num_nominatives           type=IntegerType()
  feat_f85_sent_len_variance                          key=f85_sent_len_variance         type=DoubleType()
  feat_f106_tanween_freq                              key=f106_tanween_freq             type=IntegerType()
  feat_f4_whitespace_ratio                            key=f4_whitespace_ratio           type=DoubleType()
  feat_f25_single_quotes                              key=f25_single_quotes             type=IntegerType()
  feat_f46_num_adverbs                                key=f46_num_adverbs               type=IntegerType()
  feat_f

## 3. Load preprocessed data

In [ ]:
preprocessed_df = load_checkpoint(spark, 'preprocessed')
preprocessed_df.cache()
print(f'Loaded {preprocessed_df.count():,} rows')
preprocessed_df.select('label', 'source_model', 'generation_method', 'clean_text').show(5, truncate=80)


Loaded 41,940 rows
+-----+------------+-----------------+--------------------------------------------------------------------------------+
|label|source_model|generation_method|                                                                      clean_text|
+-----+------------+-----------------+--------------------------------------------------------------------------------+
|    1|      openai|     by_polishing|صور نظام تعليم مراه اندلسيه تستند دراسه دقيقه مصادر تاريخيه وثقت حياه علماء م...|
|    1|      openai|     by_polishing|انهيار دوله موحد يعود بشكل كبير عوامل ثقافيه، تقل اهميه عوامل سياسيه عسكريه م...|
|    1|      openai|     by_polishing|جهود قاده ثوره جزائريه مرحله اولي حاسمه بحث مصادر تسليح عبر حدود غربيه، اسهمت...|
|    1|      openai|     by_polishing|مقال يناقش قضيه ضرائب شرعيه دولتي مرابط موحدين، موضحا تاثير ضرائب علاقه دوله ...|
|    1|      openai|     by_polishing|حركه انتصار حري ديمقراطيه جزائر، فتره شهدت تحول جذريه بدات حرب عالميه ثانيه و...|
+-----+------------+-

## 4. MapReduce design note (for Methodology section)

In [ ]:
print_mapreduce_design()



══════════════════════════════════════════════════════════════
  MapReduce Design — Hapax Legomena Ratio (f13)
══════════════════════════════════════════════════════════════

Job 1 — Word Count
  MAP:    (doc_id, text)  →  (word, 1)  for each word
  REDUCE: (word, [1,1,…]) →  (word, total_count)

Job 2 — Hapax Identification
  MAP:    (word, total_count) | filter count==1
                              → ("HAPAX_KEY", 1)
  REDUCE: ("HAPAX_KEY", [1,1,…]) → hapax_total

Final Calculation
  hapax_ratio = hapax_total / total_word_count

Spark equivalent:
  Job1 → df.explode + groupBy("word").count()
  Job2 → filter(count == 1).count()
  Final → hapax_count / total_count
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  MapReduce Design — Word Entropy (f22)
══════════════════════════════════════════════════════════════

Job 1 — Word Count  (same as above)

Job 2 — Entropy Calculation
  MAP:    (word, count) → p =

## 5. Extract stylometric features — three cost tiers

**Why tiers?** The 17 features fall into three cost groups:

| Tier | Features | Method | Typical time |
|------|----------|--------|-------------|
| 1 — Light | f1, f4, f13, f22, f25, f34, f85, f106, f109 | pandas_udf (Arrow) | 2–5 min |
| 2 — CAMeL | f43, f46, f55, f64, f67, f76 | pandas_udf (Arrow) | 10–20 min |
| 3 — Deep  | f88 (sentence-transformers), f97 (BERT) | batch pandas_udf | 20–60 min |

Each tier has its own GDrive checkpoint so a crashed session only re-runs
the failed tier, not all 17 features from scratch.

**`FAST_MODE = True`** (recommended for development) sets f88 and f97 to 0.0
so Tier 3 completes instantly.  Switch to `False` for the final submission run.

**How it is faster than before:**
- `pandas_udf` sends data via Apache Arrow — no pickle overhead per row
- CAMeL Tools morphology DB is loaded **once per executor process**, not per row
- BERT / sentence-transformer encodes **all sentences in a partition in one
  batched forward pass**, not one sentence at a time

In [ ]:
# ── 0. Enable Arrow-optimised pandas UDFs (must be set before any UDF runs)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "2000")
print("Arrow optimization enabled.")


from src.feature_engineering import (
    _PANDAS_UDFS, _FEATURE_REGISTRY,
    LIGHT_KEYS, CAMEL_KEYS, DEEP_KEYS,
    FEATURE_COLS, extract_features_tiered,
)

FAST_MODE = False   # ← set False to compute f88/f97 with real BERT embeddings

print(f"FAST_MODE={FAST_MODE}  (deep features {'→ 0.0 placeholder' if FAST_MODE else '→ real BERT'})")
print(f"Tier 1 keys ({len(LIGHT_KEYS)}): {LIGHT_KEYS}")
print(f"Tier 2 keys ({len(CAMEL_KEYS)}): {CAMEL_KEYS}")
print(f"Tier 3 keys ({len(DEEP_KEYS)}):  {DEEP_KEYS}")


Arrow optimization enabled.
FAST_MODE=False  (deep features → real BERT)
Tier 1 keys (9): ['f1_total_chars', 'f4_whitespace_ratio', 'f13_hapax_ratio', 'f22_word_entropy', 'f25_single_quotes', 'f34_num_sentences', 'f85_sent_len_variance', 'f106_tanween_freq', 'f109_link_frequency']
Tier 2 keys (6): ['f43_num_nouns', 'f46_num_adverbs', 'f55_noun_verb_ratio', 'f64_num_nominatives', 'f67_num_singular', 'f76_passive_sents']
Tier 3 keys (2):  ['f88_semantic_similarity', 'f97_bert_similarity']


In [ ]:
# ── Tier 1: 9 light features (pure text, no external model)

if checkpoint_exists('features_light'):
    light_df = load_checkpoint(spark, 'features_light')
    print('Loaded Tier 1 (light) from checkpoint.')
else:
    with Timer('Tier 1 — 9 light features (pandas_udf + Arrow)'):
        result = preprocessed_df   # keep existing partitioning for max parallelism
        for key in LIGHT_KEYS:
            _, _, col_name = _FEATURE_REGISTRY[key]
            result = result.withColumn(col_name, _PANDAS_UDFS[key](F.col('clean_text')))
            print(f'  ✓ {col_name}')
        save_checkpoint(result, 'features_light')
        light_df = result
    print('Tier 1 complete and checkpointed.')

light_df.cache()
print(f'Tier 1 rows: {light_df.count():,}')
t1_cols = [c for c in light_df.columns if c.startswith(FEAT_PREFIX)]
print(f'Feature columns so far: {len(t1_cols)}')


Loaded Tier 1 (light) from checkpoint.
Tier 1 rows: 41,940
Feature columns so far: 9


In [ ]:
# ── Tier 2: 6 CAMeL morphology features

from src.feature_engineering import _pudf_all_camel, CAMEL_STRUCT_TYPE, _CAMEL_COLS

if checkpoint_exists('features_camel'):
    camel_df = load_checkpoint(spark, 'features_camel')
    print('Loaded Tier 2 (CAMeL) from checkpoint.')
else:
    with Timer('Tier 2 — 6 CAMeL features (vocab-cache struct pandas_udf)'):
        # Repartition to 4: each partition builds its own vocab cache independently
        # 4 processes × ~25 000 unique tokens each × ~5 ms = ~2 min  (was 40+ min)
        result = light_df.repartition(4)

        # ONE withColumn call returns a struct with all 6 feature fields at once
        result = result.withColumn("_camel_struct", _pudf_all_camel(F.col("clean_text")))

        # Extract each field from the struct into its own top-level column
        for field in CAMEL_STRUCT_TYPE.fields:
            result = result.withColumn(field.name, F.col(f"_camel_struct.{field.name}"))

        result = result.drop("_camel_struct")
        save_checkpoint(result, 'features_camel')
        camel_df = result
    print('Tier 2 complete and checkpointed.')

camel_df.cache()
t2_cols = [c for c in camel_df.columns if c.startswith(FEAT_PREFIX)]
print(f'Tier 2 rows: {camel_df.count():,}  |  feature columns: {len(t2_cols)}')

# Quick sanity check on one CAMeL feature
camel_df.select('clean_text', *_CAMEL_COLS).show(3, truncate=60)


Loaded Tier 2 (CAMeL) from checkpoint.
Tier 2 rows: 41,940  |  feature columns: 15
+------------------------------------------------------------+------------------+--------------------+------------------------+------------------------+---------------------+----------------------+
|                                                  clean_text|feat_f43_num_nouns|feat_f46_num_adverbs|feat_f55_noun_verb_ratio|feat_f64_num_nominatives|feat_f67_num_singular|feat_f76_passive_sents|
+------------------------------------------------------------+------------------+--------------------+------------------------+------------------------+---------------------+----------------------+
|يعني بحث بدراسه دلاله نحويه وسمات جمله، يشدد اهميه انظمه ...|                30|                   0|                4.285714|                      16|                   41|                     0|
|تحلل دراسه بشكل دقيق استقلاليه ديو وطني اوقاف زكاه والذي ...|                68|                   1|                2.42857

In [ ]:
# ── Tier 3: 2 deep learning features (BERT + sentence-transformers)

if checkpoint_exists('features_stylometric'):
    features_df = load_checkpoint(spark, 'features_stylometric')
    print('Loaded full stylometric checkpoint (Tier 3 already done).')
elif FAST_MODE:
    print('FAST_MODE=True → skipping BERT/sentence-transformer, filling with 0.0')
    result = camel_df
    for key in DEEP_KEYS:
        _, ret_type, col_name = _FEATURE_REGISTRY[key]
        result = result.withColumn(col_name, F.lit(0.0).cast(ret_type))
        print(f'  ○ {col_name} = 0.0 (placeholder)')
    save_checkpoint(result, 'features_stylometric')
    features_df = result
    print('Tier 3 complete (fast mode).')
else:
    with Timer('Tier 3 — 2 deep learning features (batch pandas_udf)'):
        result = camel_df.repartition(2)   # 2 partitions → models load 2× only
        for key in DEEP_KEYS:
            _, _, col_name = _FEATURE_REGISTRY[key]
            result = result.withColumn(col_name, _PANDAS_UDFS[key](F.col('clean_text')))
            print(f'  ✓ {col_name}')
        save_checkpoint(result, 'features_stylometric')
        features_df = result
    print('Tier 3 complete and checkpointed.')

features_df.cache()
print(f'\nAll features: {features_df.count():,} rows')

available = [c for c in features_df.columns if c.startswith(FEAT_PREFIX)]
missing   = [c for c in FEATURE_COLS if c not in features_df.columns]
print(f'Feature columns present : {len(available)} / {len(FEATURE_COLS)}')
if missing:
    print('WARNING — missing columns:', missing)
else:
    print(' All 17 feature columns present.')


Loaded full stylometric checkpoint (Tier 3 already done).

All features: 41,940 rows
Feature columns present : 17 / 17
✅ All 17 feature columns present.


## 6. Build TF-IDF features (Task 3.2)

In [ ]:
if checkpoint_exists('features_tfidf'):
    tfidf_df = load_checkpoint(spark, 'features_tfidf')
    print('Loaded TF-IDF checkpoint.')
else:
    tfidf_pipeline = build_tfidf_pipeline(
        num_features=20_000, min_df=2,
        input_col='clean_text', output_col='tfidf_features',
    )
    with Timer('TF-IDF fit + transform'):
        tfidf_model = tfidf_pipeline.fit(features_df)
        tfidf_df    = tfidf_model.transform(features_df).drop('_tokens', '_tf_raw')
        save_checkpoint(tfidf_df, 'features_tfidf')

    tfidf_model.write().overwrite().save(str(MODELS_DIR / 'tfidf_pipeline'))
    print('TF-IDF pipeline saved to GDrive.')

print('tfidf_features column present:', 'tfidf_features' in tfidf_df.columns)
print(f'Rows: {tfidf_df.count():,}')


TF-IDF pipeline saved to GDrive.
tfidf_features column present: True
Rows: 41,940


## 7. Optional — Word2Vec embeddings (Task 3.2)

In [ ]:
RUN_WORD2VEC = False   # ← set True to include Word2Vec in modeling

if RUN_WORD2VEC:
    if checkpoint_exists('features_w2v'):
        w2v_df = load_checkpoint(spark, 'features_w2v')
        print('Loaded Word2Vec checkpoint.')
    else:
        w2v_pipeline = build_word2vec_pipeline(vector_size=100, min_count=2)
        with Timer('Word2Vec fit + transform'):
            w2v_model = w2v_pipeline.fit(tfidf_df)
            w2v_df    = w2v_model.transform(tfidf_df)
            save_checkpoint(w2v_df, 'features_w2v')
        w2v_model.write().overwrite().save(str(MODELS_DIR / 'w2v_pipeline'))
        print('Word2Vec pipeline saved.')
else:
    print('Word2Vec skipped (RUN_WORD2VEC=False).')


Word2Vec skipped (RUN_WORD2VEC=False).


## 8. Stratified train / validation / test split (Task 3.3)

In [ ]:
if (checkpoint_exists('split_train') and checkpoint_exists('split_val')
        and checkpoint_exists('split_test')):
    train_df = load_checkpoint(spark, 'split_train')
    val_df   = load_checkpoint(spark, 'split_val')
    test_df  = load_checkpoint(spark, 'split_test')
    print('Loaded split checkpoints.')
else:
    train_df, val_df, test_df = stratified_split(
        tfidf_df, train_ratio=0.70, val_ratio=0.15, seed=42
    )
    save_checkpoint(train_df, 'split_train')
    save_checkpoint(val_df,   'split_val')
    save_checkpoint(test_df,  'split_test')
    print('Splits saved.')

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n   = df.count()
    pos = df.filter(F.col('label') == 1).count()
    print(f'{name:5s}: {n:6,} rows  AI={pos/n*100:.1f}%')


Splits saved.
Train: 29,669 rows  AI=79.9%
Val  :  6,091 rows  AI=80.3%
Test :  6,180 rows  AI=80.0%


## 9. Assemble final feature vector

In [ ]:
for split_name, df_in in [
    ('split_train_assembled', train_df),
    ('split_val_assembled',   val_df),
    ('split_test_assembled',  test_df),
]:
    if checkpoint_exists(split_name):
        print(f'{split_name}: already exists, skipping.')
    else:
        with Timer(f'Assemble {split_name}'):
            assembled = assemble_feature_vector(df_in, tfidf_col='tfidf_features')
            save_checkpoint(assembled, split_name)
        print(f'{split_name}: assembled and saved.')


split_train_assembled: assembled and saved.
split_val_assembled: assembled and saved.
split_test_assembled: assembled and saved.


## 10. Feature statistics

In [ ]:
assembled_train = load_checkpoint(spark, 'split_train_assembled')
print('features column present:', 'features' in assembled_train.columns)

feat_stats = (
    assembled_train
    .select([F.col(c).cast('double').alias(c) for c in FEATURE_COLS if c in assembled_train.columns])
    .describe()
)
feat_stats.show(truncate=False)


features column present: True
+-------+-------------------+---------------------+------------------+------------------------+--------------------------+----------------------+------------------------+----------------------+--------------------+---------------------+---------------------+-------------------+--------------------+----------------------+------------------------+----------------------+-------------------+
|summary|feat_f1_total_chars|feat_f22_word_entropy|feat_f43_num_nouns|feat_f64_num_nominatives|feat_f85_sent_len_variance|feat_f106_tanween_freq|feat_f4_whitespace_ratio|feat_f25_single_quotes|feat_f46_num_adverbs|feat_f67_num_singular|feat_f88_semantic_sim|feat_f109_link_freq|feat_f13_hapax_ratio|feat_f34_num_sentences|feat_f55_noun_verb_ratio|feat_f76_passive_sents|feat_f97_bert_sim  |
+-------+-------------------+---------------------+------------------+------------------------+--------------------------+----------------------+------------------------+------------------